# Qwen на экзаменационном наборе

Подключить через **Add Input** один Kaggle Dataset `production_dataset` с выданным файлом `dataset.jsonl` (production_dataset). В каждой строке нужны `photo_id` и `photo_url`.

Настройки победившего Qwen вставляются в ячейку 2 один раз из локального файла `results/experiments/qwen/qwen.experiment.json`.


## 1. Читаем выданный датасет


In [ ]:
import json
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DATASET_PATH = None

if DATASET_PATH is None:
    candidates = sorted(
        path for name in ("dataset.jsonl", "production_dataset.jsonl", "production_dataset.json")
        for path in INPUT.rglob(name) if path.is_file()
    )
    production_files = [
        path for path in candidates
        if any(part.lower().replace("_", "-") == "production-dataset"
               for part in path.relative_to(INPUT).parts[:-1])
    ]
    choices = production_files or candidates
    if len(choices) != 1:
        raise ValueError(
            "Подключи production_dataset через Add Input или задай DATASET_PATH. "
            f"Найденные файлы: {choices}"
        )
    DATASET_PATH = choices[0]
else:
    DATASET_PATH = Path(DATASET_PATH)

text = DATASET_PATH.read_text(encoding="utf-8-sig")
if text.lstrip().startswith("["):
    records = json.loads(text)
else:
    records = [json.loads(line) for line in text.splitlines() if line.strip()]

contest = {}
for index, row in enumerate(records, 1):
    photo_id = row.get("photo_id")
    source = row.get("photo_url")
    if type(photo_id) is not int or photo_id <= 0 or photo_id in contest:
        raise ValueError(f"Запись {index}: неверный или повторный photo_id {photo_id!r}")
    if not isinstance(source, str) or not source.startswith(("https://", "http://")):
        raise ValueError(f"Запись {index}: нет публичной ссылки photo_url")
    contest[photo_id] = {"image": source.strip()}

if len(contest) != 400:
    raise ValueError(f"Ожидается 400 выданных фото. Найдено {len(contest)}")
print(f"Загружено {len(contest)} фото из {DATASET_PATH}")


Загружено 400 фото из /kaggle/input/datasets/hmossya/production-dataset/production_dataset.jsonl


## 2. Вставляем настройки победившего прогона

In [27]:
WINNER = "qwen__strict_negative_prompt"

WINNER_SETTINGS = {'config': {'name': 'qwen__strict_negative_prompt',
  'backend': 'qwen',
  'model_id': 'Qwen/Qwen3-VL-2B-Instruct',
  'prompt': 'Detect every visible adult butterfly or moth. Give each individual one tight bounding box covering only its visible parts. Ignore other insects, caterpillars, pupae, and shadows. Return only a JSON array of objects with "bbox_2d": [x_min, y_min, x_max, y_max], using integer coordinates from 0 to 1000 relative to the full image. Return [] if no adult butterfly or moth is visible. Include a box only if the visible anatomy clearly identifies an adult butterfly or moth. On a flower-only image or an image containing only other insects, return [].',
  'revision': '89644892e4d85e24eaac8bacfd4f463576704203',
  'candidate_threshold': 0.0,
  'threshold': 0.0,
  'text_threshold': 0.25,
  'max_new_tokens': 512,
  'image_max_side': 1024,
  'load_in_4bit': False},
 'device': 'cuda',
 'gpu': 'Tesla T4',
 'torch': '2.11.0+cu130',
 'transformers': '4.57.1',
 'manifest_hash': 'ff95168fc41cc3fb8219d65ca9720d447cccd4cc0ca657fe88906a82b0fbee4b'}

winner_config = WINNER_SETTINGS["config"]
if winner_config["name"] != WINNER or winner_config["backend"] != "qwen":
    raise ValueError("Вставь настройки qwen__strict_negative_prompt из эксперимента.")
if not winner_config.get("revision") or winner_config["revision"] == "main":
    raise ValueError("В настройках победителя должна быть конкретная ревизия весов.")

print("Модель:", winner_config["model_id"])
print("Ревизия:", winner_config["revision"])
print("Промпт:", winner_config["prompt"])


Модель: Qwen/Qwen3-VL-2B-Instruct
Ревизия: 89644892e4d85e24eaac8bacfd4f463576704203
Промпт: Detect every visible adult butterfly or moth. Give each individual one tight bounding box covering only its visible parts. Ignore other insects, caterpillars, pupae, and shadows. Return only a JSON array of objects with "bbox_2d": [x_min, y_min, x_max, y_max], using integer coordinates from 0 to 1000 relative to the full image. Return [] if no adult butterfly or moth is visible. Include a box only if the visible anatomy clearly identifies an adult butterfly or moth. On a flower-only image or an image containing only other insects, return [].


## 3. Готовим окружение Kaggle


In [28]:
import os
import subprocess
import sys

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")

packages = [
    f"transformers=={WINNER_SETTINGS['transformers']}",
    "huggingface-hub>=0.34,<1",
    "accelerate>=1,<2",
]
if winner_config.get("load_in_4bit"):
    packages.append("bitsandbytes")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Включи GPU в настройках Kaggle.")
device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))


GPU: Tesla T4


## 4. Функции загрузки модели, инференса и проверки посылки


In [29]:
import gc
import hashlib
import math
import re
from dataclasses import asdict, dataclass, replace
from io import BytesIO
from time import perf_counter
from urllib.request import urlopen

from PIL import Image, ImageOps


@dataclass(frozen=True)
class Config:
    name: str
    backend: str
    model_id: str
    prompt: str | tuple[str, ...]
    revision: str = "main"
    candidate_threshold: float = 0.0
    threshold: float = 0.0
    text_threshold: float = 0.25
    max_new_tokens: int = 512
    image_max_side: int = 1024
    load_in_4bit: bool = False


def normalized_box(coords, width, height):
    """Convert XYXY coordinates to the contest format; ignore empty boxes."""
    x1, y1, x2, y2 = coords
    if not all(math.isfinite(value) for value in coords):
        raise ValueError("Non-finite box coordinates")
    left, right = (min(1.0, max(0.0, x / width)) for x in (x1, x2))
    top, bottom = (min(1.0, max(0.0, y / height)) for y in (y1, y2))
    if right <= left or bottom <= top:
        return None
    return dict(left=left, top=top, width=right-left, height=bottom-top)


def parse_qwen(text):
    """Parse the whole response, accepting an optional Markdown JSON fence."""
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text.strip(), re.S | re.I)
    items = json.loads(fenced.group(1) if fenced else text)
    if not isinstance(items, list):
        raise ValueError("Expected a JSON list")
    boxes = []
    for item in items:
        coords = item.get("bbox_2d") if isinstance(item, dict) else None
        if not isinstance(coords, list) or len(coords) != 4 or not all(
            type(value) in (int, float) and 0 <= value <= 1000 for value in coords
        ):
            raise ValueError("Expected four bbox_2d coordinates in [0, 1000]")
        box = normalized_box(coords, 1000, 1000)
        if box is None:
            raise ValueError("Qwen returned an empty or reversed box")
        boxes.append(box)
    return boxes


class QwenResponseError(ValueError):
    def __init__(self, message, raw_response):
        super().__init__(message)
        self.raw_response = raw_response


def load_detector(config, device):
    import torch
    from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
    from transformers import BitsAndBytesConfig

    processor = AutoProcessor.from_pretrained(config.model_id, revision=config.revision)
    if config.backend == "qwen":
        from transformers import Qwen3VLForConditionalGeneration
        dtype = torch.float16 if device.startswith("cuda") else torch.float32
        options = {"dtype": dtype}
        if config.load_in_4bit:
            if not device.startswith("cuda"):
                raise ValueError("This 4-bit configuration requires a CUDA device")
            options["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype,
            )
            options["device_map"] = {"": device}
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            config.model_id, revision=config.revision, **options
        )
        if not config.load_in_4bit:
            model.to(device)
    else:
        model = AutoModelForZeroShotObjectDetection.from_pretrained(
            config.model_id, revision=config.revision
        ).to(device)
    return processor, model.eval()


def predict_one(detector, config, image, device):
    """Return candidates (each with a box and optional score) and model text."""
    import torch
    from PIL import Image

    processor, model = detector
    with torch.inference_mode():
        if config.backend == "qwen":
            resized = image.copy()
            resized.thumbnail(
                (config.image_max_side, config.image_max_side), Image.Resampling.LANCZOS
            )
            messages = [{"role": "user", "content": [
                {"type": "image", "image": resized},
                {"type": "text", "text": config.prompt},
            ]}]
            chat_text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(
                text=[chat_text], images=[resized], return_tensors="pt"
            ).to(device)
            ids = model.generate(
                **inputs, max_new_tokens=config.max_new_tokens, do_sample=False
            )
            text = processor.batch_decode(
                ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
            )[0]
            try:
                boxes = parse_qwen(text)
            except ValueError as error:
                raise QwenResponseError(str(error), text) from error
            return [{"box": box, "score": None} for box in boxes], text

        size = [(image.height, image.width)]
        query = config.prompt if config.backend == "dino" else [list(config.prompt)]
        inputs = processor(images=image, text=query, return_tensors="pt").to(device)
        outputs = model(**inputs)
        if config.backend == "dino":
            result = processor.post_process_grounded_object_detection(
                outputs, input_ids=inputs.input_ids, target_sizes=size,
                threshold=config.candidate_threshold, text_threshold=config.text_threshold,
            )[0]
        else:
            result = processor.image_processor.post_process_object_detection(
                outputs, target_sizes=size, threshold=config.candidate_threshold
            )[0]

    candidates = []
    for coords, score in zip(result["boxes"].tolist(), result["scores"].tolist()):
        box = normalized_box(coords, image.width, image.height)
        if box is not None:
            candidates.append({"box": box, "score": score})
    return candidates, None


def select_boxes(candidates, threshold):
    return [
        candidate["box"] for candidate in candidates
        if candidate["score"] is None or candidate["score"] >= threshold
    ]


def open_image(source):
    if source.startswith(("https://", "http://")):
        with urlopen(source, timeout=30) as response:
            source = BytesIO(response.read())
    with Image.open(source) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def read_jsonl(path):
    rows = {}
    with Path(path).open(encoding="utf-8-sig") as file:
        for line in file:
            if not line.strip():
                continue
            row = json.loads(line)
            record_id = row.get("photo_id")
            if type(record_id) is not int or record_id in rows:
                raise ValueError(f"{path}: неверный или повторный photo_id {record_id!r}")
            rows[record_id] = row
    return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    temp.replace(path)


def read_settings(raw_path):
    path = Path(raw_path).with_suffix(".settings.json")
    return json.loads(path.read_text(encoding="utf-8"))


def run_model(config, manifest, raw_path, device):
    """Сохраняет кандидатов после каждого фото. При повторе обрабатывает сбои."""
    import torch
    import transformers
    from huggingface_hub import model_info

    raw_path = Path(raw_path)
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    settings_path = raw_path.with_suffix(".settings.json")
    previous = read_settings(raw_path) if settings_path.exists() else None
    if raw_path.exists() and previous is None:
        raise ValueError(f"Для {raw_path} отсутствует файл настроек")

    if config.revision == "main":
        revision = previous["config"]["revision"] if previous else model_info(config.model_id).sha
        config = replace(config, revision=revision)

    settings = {
        "config": asdict(config),
        "device": device,
        "gpu": torch.cuda.get_device_name(device) if device.startswith("cuda") else None,
        "torch": str(torch.__version__),
        "transformers": transformers.__version__,
        "manifest_hash": hashlib.sha256(
            json.dumps(manifest, sort_keys=True).encode("utf-8")
        ).hexdigest(),
    }
    settings_text = json.dumps(settings, ensure_ascii=False, indent=2)
    if previous is not None and previous != json.loads(settings_text):
        raise ValueError("Настройки или набор фото изменились. Укажи новый путь для результатов.")
    if previous is None:
        settings_path.write_text(settings_text, encoding="utf-8")

    results = read_jsonl(raw_path) if raw_path.exists() else {}
    pending = [pid for pid in manifest if pid not in results or results[pid]["error"]]
    if not pending:
        return results

    print(f"Загружаю {config.model_id} на {device}", flush=True)
    detector = load_detector(config, device)
    print(f"Веса загружены. Осталось фото: {len(pending)}", flush=True)
    try:
        for index, photo_id in enumerate(pending, 1):
            previous_attempts = results.get(photo_id, {}).get("attempt", 0)
            record = {"photo_id": photo_id, "attempt": previous_attempts + 1}
            try:
                image = open_image(manifest[photo_id]["image"])
                if device.startswith("cuda"):
                    torch.cuda.synchronize(device)
                started = perf_counter()
                candidates, text = predict_one(detector, config, image, device)
                if device.startswith("cuda"):
                    torch.cuda.synchronize(device)
                record.update(candidates=candidates, raw_response=text,
                              seconds=perf_counter() - started, error=None)
            except Exception as error:
                record.update(candidates=None, seconds=None,
                              raw_response=getattr(error, "raw_response", None),
                              error=f"{type(error).__name__}: {error}")

            if record["error"] or previous_attempts:
                log_path = raw_path.with_suffix(".errors.jsonl")
                with log_path.open("a", encoding="utf-8") as log:
                    log.write(json.dumps(record, ensure_ascii=False) + "\n")

            results[photo_id] = record
            write_jsonl(raw_path, results.values())
            # if index == 1 or index % 10 == 0 or index == len(pending):
            print(f"{config.name}: {index}/{len(pending)}", flush=True)
    finally:
        del detector
        gc.collect()
        if device.startswith("cuda"):
            torch.cuda.empty_cache()
    return results


def export_predictions(raw, manifest, threshold, path):
    if set(raw) != set(manifest):
        raise ValueError("ID результатов и манифеста должны совпадать")
    rows = {}
    for photo_id in manifest:
        record = raw[photo_id]
        boxes = None if record["error"] else select_boxes(record["candidates"], threshold)
        rows[photo_id] = {"photo_id": photo_id, "boxes": boxes, "error": record["error"]}
    write_jsonl(path, rows.values())
    return rows


def check_boxes(boxes):
    if not isinstance(boxes, list):
        raise ValueError("boxes должен быть списком")
    for box in boxes:
        x, y, w, h = (box[key] for key in ("left", "top", "width", "height"))
        if not all(type(v) in (float, int) and math.isfinite(v) for v in (x, y, w, h)):
            raise ValueError(f"Неверные координаты: {box}")
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                and x+w <= 1+1e-9 and y+h <= 1+1e-9):
            raise ValueError(f"Рамка выходит за границы или имеет нулевой размер: {box}")


def validate_submission(path, manifest):
    rows = read_jsonl(path)
    if len(manifest) != 400 or set(rows) != set(manifest):
        raise ValueError("В посылке должны быть ровно 400 выданных photo_id")
    if Path(path).stat().st_size > 5 * 1024**2:
        raise ValueError("Посылка больше 5 МиБ")
    for photo_id, row in rows.items():
        if row.get("error") is not None:
            raise ValueError(f"Не обработана фотография {photo_id}: {row['error']}")
        check_boxes(row["boxes"])


## 5. Прогоняем Qwen на 400 фото

Прогресс сохраняется после каждого фото. Если какой-то запрос упадёт, после устранения причины повтори эту ячейку в той же сессии. Успешные записи возьмутся из сохранённого файла. Если выданная ссылка недоступна, сообщи преподавателю и сохрани исходную ссылку.


In [30]:
config = Config(**winner_config)
raw_path = WORK / "contest" / f"{WINNER}.raw.jsonl"

raw = run_model(config, contest, raw_path, device)
failures = [(pid, row["error"]) for pid, row in raw.items() if row["error"]]
print(f"Записей: {len(raw)} | ошибок: {len(failures)}")

for photo_id, error in failures[:10]:
    print(f"{photo_id}: {error}")
if failures:
    print(f"Журнал ошибок: {raw_path.with_suffix('.errors.jsonl')}")


Загружаю Qwen/Qwen3-VL-2B-Instruct на cuda
Веса загружены. Осталось фото: 400
qwen__strict_negative_prompt: 1/400
qwen__strict_negative_prompt: 2/400
qwen__strict_negative_prompt: 3/400
qwen__strict_negative_prompt: 4/400
qwen__strict_negative_prompt: 5/400
qwen__strict_negative_prompt: 6/400
qwen__strict_negative_prompt: 7/400
qwen__strict_negative_prompt: 8/400
qwen__strict_negative_prompt: 9/400
qwen__strict_negative_prompt: 10/400
qwen__strict_negative_prompt: 11/400
qwen__strict_negative_prompt: 12/400
qwen__strict_negative_prompt: 13/400
qwen__strict_negative_prompt: 14/400
qwen__strict_negative_prompt: 15/400
qwen__strict_negative_prompt: 16/400
qwen__strict_negative_prompt: 17/400
qwen__strict_negative_prompt: 18/400
qwen__strict_negative_prompt: 19/400
qwen__strict_negative_prompt: 20/400
qwen__strict_negative_prompt: 21/400
qwen__strict_negative_prompt: 22/400
qwen__strict_negative_prompt: 23/400
qwen__strict_negative_prompt: 24/400
qwen__strict_negative_prompt: 25/400
qwen__

## 6. Проверяем и скачиваем JSONL для Контеста


In [31]:
from IPython.display import FileLink, display

if failures:
    raise RuntimeError(f"Осталось ошибок: {len(failures)}. Повтори прогон после их устранения.")

draft_path = WORK / "submission.pending.jsonl"
submission_path = WORK / "submission.jsonl"
export_predictions(raw, contest, config.threshold, draft_path)
validate_submission(draft_path, contest)
draft_path.replace(submission_path)

print(f"Готово: {submission_path} | {submission_path.stat().st_size:,} байт")
display(FileLink("submission.jsonl"))


Готово: /kaggle/working/submission.jsonl | 37,551 байт


/kaggle/working/submission.jsonl